In [8]:
import pandas as pd

In [9]:
df = pd.read_csv('train.csv')
df

,id,subject,email,spam
0,0,Subject: A&L Daily to be auctioned in bankrupt...,URL: http://boingboing.net/#85534171\n Date: N...,0
1,1,"Subject: Wired: ""Stronger ties between ISPs an...",URL: http://scriptingnews.userland.com/backiss...,0
2,2,Subject: It's just too small ...,<HTML>\n <HEAD>\n </HEAD>\n <BODY>\n <FONT SIZ...,1
3,3,Subject: liberal defnitions\n,Depends on how much over spending vs. how much...,0
4,4,Subject: RE: [ILUG] Newbie seeks advice - Suse...,hehe sorry but if you hit caps lock twice the ...,0
...,...,...,...,...
8343,8343,Subject: Re: ALSA (almost) made easy\n,"Thanks for this, I'm going to give them anothe...",0
8344,8344,Subject: Re: Goodbye Global Warming\n,Thanks for the link - I'm fascinated by archae...,0
8345,8345,Subject: hello\n,WE NEED HELP. We are a 14 year old fortune 50...,1
8346,8346,Subject: Your application is below. Expires Ju...,<html>\n \n \n <HEAD> \n <META charset=3DUTF-8...,1


In [10]:
test_df = pd.read_csv('test.csv')
test_df

,id,subject,email
0,0,Subject: CERT Advisory CA-2002-21 Vulnerabilit...,\n \n -----BEGIN PGP SIGNED MESSAGE-----\n \n ...
1,1,Subject: ADV: Affordable Life Insurance ddbfk\n,Low-Cost Term-Life Insurance!\n SAVE up to 70%...
2,2,Subject: CAREER OPPORTUNITY. WORK FROM HOME\n,------=_NextPart_000_00A0_03E30A1A.B1804B54\n ...
3,3,Subject: Marriage makes both sexes happy\n,"URL: http://www.newsisfree.com/click/-3,848315..."
4,4,Subject: Re: [SAtalk] SA very slow (hangs?) on...,On Thursday 29 August 2002 16:39 CET Mike Burg...
...,...,...,...
995,995,Subject: Re: Alsa/Redhat 8 compatability\n,"Once upon a time, Brian wrote :\n \n > \n > ..."
996,996,Subject: Re: Goodbye Global Warming\n,\n --]> A Green once said that if the Spotted ...
997,997,Subject: Re: Entrepreneurs\n,"On Fri, 23 Aug 2002, Robert Harley wrote:\n \n..."
998,998,Subject: Re: [ILUG] slashdot EW Dijkstra humor\n,JPL suggested:\n > Recursion is only truely u...


In [11]:
df.info()

<class 'pandas.DataFrame'>
RangeIndex: 8348 entries, 0 to 8347
Data columns (total 4 columns):
 #   Column   Non-Null Count  Dtype
---  ------   --------------  -----
 0   id       8348 non-null   int64
 1   subject  8342 non-null   str  
 2   email    8348 non-null   str  
 3   spam     8348 non-null   int64
dtypes: int64(2), str(2)
memory usage: 261.0 KB


In [12]:
df.head()

,id,subject,email,spam
0,0,Subject: A&L Daily to be auctioned in bankrupt...,URL: http://boingboing.net/#85534171\n Date: N...,0
1,1,"Subject: Wired: ""Stronger ties between ISPs an...",URL: http://scriptingnews.userland.com/backiss...,0
2,2,Subject: It's just too small ...,<HTML>\n <HEAD>\n </HEAD>\n <BODY>\n <FONT SIZ...,1
3,3,Subject: liberal defnitions\n,Depends on how much over spending vs. how much...,0
4,4,Subject: RE: [ILUG] Newbie seeks advice - Suse...,hehe sorry but if you hit caps lock twice the ...,0


In [13]:
df.isna().sum()

id         0
subject    6
email      0
spam       0
dtype: int64

In [14]:
df['text'] = df['subject'].fillna('') + df['email'].fillna('')
test_df['text'] = test_df['subject'].fillna('') +test_df['email'].fillna('')

In [15]:
import re

def clean_text(text):
    text = text.lower()
    text = re.sub(r'<.*?>', '', text)  # Remove HTML tags
    text = re.sub(r'[^a-z\s]', '', text)  # Remove non-alphabet characters
    text = re.sub(r'\s+', ' ', text).strip()
    return text

df['clean_text'] = df['text'].apply(clean_text)
test_df['clean_text'] = test_df['text'].apply(clean_text)


In [16]:
df['spam'].value_counts()

spam
0    6208
1    2140
Name: count, dtype: int64

In [17]:
df.columns

Index(['id', 'subject', 'email', 'spam', 'text', 'clean_text'], dtype='str')

In [18]:
from sklearn.feature_extraction.text import TfidfVectorizer
vectorizer = TfidfVectorizer()

vectorizer = TfidfVectorizer(
    max_features=5000,    
    ngram_range=(1, 2)    
)

In [19]:
from sklearn.model_selection import train_test_split

X = vectorizer.fit_transform(df['clean_text'])
y = df['spam']

X_train, X_val, y_train, y_val = train_test_split(
    X, y,
    test_size=0.2,       
    random_state=42,
    stratify=y           
)

In [20]:
# from sklearn.linear_model import LogisticRegression
# from sklearn.metrics import classification_report

# model = LogisticRegression(max_iter=1000)
# model.fit(X_train, y_train)
# preds = model.predict(X_val)


# print(classification_report(y_val,preds))


              precision    recall  f1-score   support

           0       0.98      0.99      0.99      1242
           1       0.96      0.95      0.96       428

    accuracy                           0.98      1670
   macro avg       0.97      0.97      0.97      1670
weighted avg       0.98      0.98      0.98      1670



In [30]:
from sklearn.naive_bayes import MultinomialNB

nb_model = MultinomialNB()
nb_model.fit(X_train, y_train)

preds = nb_model.predict(X_val)
print(classification_report(y_val,preds))


              precision    recall  f1-score   support

           0       0.98      0.98      0.98      1242
           1       0.93      0.95      0.94       428

    accuracy                           0.97      1670
   macro avg       0.96      0.96      0.96      1670
weighted avg       0.97      0.97      0.97      1670



We find that the logistic regression model is doing good so we further hypertune it but since it is an NLP task we better move forward with naive bayes

In [25]:
param_grid = {
    'C': [0.01, 0.1, 1, 10],          
    'solver': ['liblinear', 'lbfgs'],
    'penalty': ['l2'],                
    'max_iter': [100, 200, 500]
}

In [26]:
from sklearn.model_selection import GridSearchCV
from sklearn.linear_model import LogisticRegression

model = LogisticRegression()
grid_search = GridSearchCV(model, param_grid, cv=5, scoring='f1', n_jobs=-1, verbose=1)
grid_search.fit(X_train, y_train)

Fitting 5 folds for each of 24 candidates, totalling 120 fits


c:\Users\Acer\Desktop\ai-ml project\Gmail-spam-filter\venv\Lib\site-packages\sklearn\linear_model\_logistic.py:1135: FutureWarning: 'penalty' was deprecated in version 1.8 and will be removed in 1.10. To avoid this warning, leave 'penalty' set to its default value and use 'l1_ratio' or 'C' instead. Use l1_ratio=0 instead of penalty='l2', l1_ratio=1 instead of penalty='l1', and C=np.inf instead of penalty=None.
  warnings.warn(


,"estimator estimator: estimator objectThis is assumed to implement the scikit-learn estimator interface.Either estimator needs to provide a ``score`` function,or ``scoring`` must be passed.",LogisticRegression()
,"param_grid param_grid: dict or list of dictionariesDictionary with parameters names (`str`) as keys and lists ofparameter settings to try as values, or a list of suchdictionaries, in which case the grids spanned by each dictionaryin the list are explored. This enables searching over any sequenceof parameter settings.","{'C': [0.01, 0.1, ...], 'max_iter': [100, 200, ...], 'penalty': ['l2'], 'solver': ['liblinear', 'lbfgs']}"
,"scoring scoring: str, callable, list, tuple or dict, default=NoneStrategy to evaluate the performance of the cross-validated model onthe test set.If `scoring` represents a single score, one can use:- a single string (see :ref:`scoring_string_names`);- a callable (see :ref:`scoring_callable`) that returns a single value;- `None`, the `estimator`'s :ref:`default evaluation criterion ` is used.If `scoring` represents multiple scores, one can use:- a list or tuple of unique strings;- a callable returning a dictionary where the keys are the metric names and the values are the metric scores;- a dictionary with metric names as keys and callables as values.See :ref:`multimetric_grid_search` for an example.",'f1'
,"n_jobs n_jobs: int, default=NoneNumber of jobs to run in parallel.``None`` means 1 unless in a :obj:`joblib.parallel_backend` context.``-1`` means using all processors. See :term:`Glossary `for more details... versionchanged:: v0.20 `n_jobs` default changed from 1 to None",-1
,"refit refit: bool, str, or callable, default=TrueRefit an estimator using the best found parameters on the wholedataset.For multiple metric evaluation, this needs to be a `str` denoting thescorer that would be used to find the best parameters for refittingthe estimator at the end.Where there are considerations other than maximum score inchoosing a best estimator, ``refit`` can be set to a function whichreturns the selected ``best_index_`` given ``cv_results_``. In thatcase, the ``best_estimator_`` and ``best_params_`` will be setaccording to the returned ``best_index_`` while the ``best_score_``attribute will not be available.The refitted estimator is made available at the ``best_estimator_``attribute and permits using ``predict`` directly on this``GridSearchCV`` instance.Also for multiple metric evaluation, the attributes ``best_index_``,``best_score_`` and ``best_params_`` will only be available if``refit`` is set and all of them will be determined w.r.t this specificscorer.See ``scoring`` parameter to know more about multiple metricevaluation.See :ref:`sphx_glr_auto_examples_model_selection_plot_grid_search_digits.py`to see how to design a custom selection strategy using a callablevia `refit`.See :ref:`this example`for an example of how to use ``refit=callable`` to balance modelcomplexity and cross-validated score... versionchanged:: 0.20 Support for callable added.",True
,"cv cv: int, cross-validation generator or an iterable, default=NoneDetermines the cross-validation splitting strategy.Possible inputs for cv are:- None, to use the default 5-fold cross validation,- integer, to specify the number of folds in a `(Stratified)KFold`,- :term:`CV splitter`,- An iterable yielding (train, test) splits as arrays of indices.For integer/None inputs, if the estimator is a classifier and ``y`` iseither binary or multiclass, :class:`StratifiedKFold` is used. In allother cases, :class:`KFold` is used. These splitters are instantiatedwith `shuffle=False` so the splits will be the same across calls.Refer :ref:`User Guide ` for the variouscross-validation strategies that can be used here... versionchanged:: 0.22 ``cv`` default value if None changed from 3-fold to 5-fold.",5
,"verbose verbose: intControls the verbosity: the higher, the more messages.- >1 : the computation time for each fold and parameter candidate is displayed;- >2

In [27]:
print( grid_search.best_params_)
print( grid_search.best_score_)

{'C': 10, 'max_iter': 100, 'penalty': 'l2', 'solver': 'lbfgs'}
0.9782655256966699


In [28]:
model = LogisticRegression(C=10, max_iter=100, penalty='l2', solver='liblinear')
model.fit(X_train,y_train)

c:\Users\Acer\Desktop\ai-ml project\Gmail-spam-filter\venv\Lib\site-packages\sklearn\linear_model\_logistic.py:1135: FutureWarning: 'penalty' was deprecated in version 1.8 and will be removed in 1.10. To avoid this warning, leave 'penalty' set to its default value and use 'l1_ratio' or 'C' instead. Use l1_ratio=0 instead of penalty='l2', l1_ratio=1 instead of penalty='l1', and C=np.inf instead of penalty=None.
  warnings.warn(


,"penalty penalty: {'l1', 'l2', 'elasticnet', None}, default='l2'Specify the norm of the penalty:- `None`: no penalty is added;- `'l2'`: add a L2 penalty term and it is the default choice;- `'l1'`: add a L1 penalty term;- `'elasticnet'`: both L1 and L2 penalty terms are added... warning:: Some penalties may not work with some solvers. See the parameter `solver` below, to know the compatibility between the penalty and solver... versionadded:: 0.19 l1 penalty with SAGA solver (allowing 'multinomial' + L1).. deprecated:: 1.8 `penalty` was deprecated in version 1.8 and will be removed in 1.10. Use `l1_ratio` instead. `l1_ratio=0` for `penalty='l2'`, `l1_ratio=1` for `penalty='l1'` and `l1_ratio` set to any float between 0 and 1 for `'penalty='elasticnet'`.",'l2'
,"C C: float, default=1.0Inverse of regularization strength; must be a positive float.Like in support vector machines, smaller values specify strongerregularization. `C=np.inf` results in unpenalized logistic regression.For a visual example on the effect of tuning the `C` parameterwith an L1 penalty, see::ref:`sphx_glr_auto_examples_linear_model_plot_logistic_path.py`.",10
,"l1_ratio l1_ratio: float, default=0.0The Elastic-Net mixing parameter, with `0 <= l1_ratio <= 1`. Setting`l1_ratio=1` gives a pure L1-penalty, setting `l1_ratio=0` a pure L2-penalty.Any value between 0 and 1 gives an Elastic-Net penalty of the form`l1_ratio * L1 + (1 - l1_ratio) * L2`... warning:: Certain values of `l1_ratio`, i.e. some penalties, may not work with some solvers. See the parameter `solver` below, to know the compatibility between the penalty and solver... versionchanged:: 1.8 Default value changed from None to 0.0... deprecated:: 1.8 `None` is deprecated and will be removed in version 1.10. Always use `l1_ratio` to specify the penalty type.",0.0
,"dual dual: bool, default=FalseDual (constrained) or primal (regularized, see also:ref:`this equation `) formulation. Dual formulationis only implemented for l2 penalty with liblinear solver. Prefer `dual=False`when n_samples > n_features.",False
,"tol tol: float, default=1e-4Tolerance for stopping criteria.",0.0001
,"fit_intercept fit_intercept: bool, default=TrueSpecifies if a constant (a.k.a. bias or intercept) should beadded to the decision function.",True
,"intercept_scaling intercept_scaling: float, default=1Useful only when the solver `liblinear` is usedand `self.fit_intercept` is set to `True`. In this case, `x` becomes`[x, self.intercept_scaling]`,i.e. a ""synthetic"" feature with constant value equal to`intercept_scaling` is appended to the instance vector.The intercept becomes``intercept_scaling * synthetic_feature_weight``... note:: The synthetic feature weight is subject to L1 or L2 regularization as all other features. To lessen the effect of regularization on synthetic feature weight (and therefore on the intercept) `intercept_scaling` has to be increased.",1
,"class_weight class_weight: dict or 'balanced', default=NoneWeights associated with classes in the form ``{class_label: weight}``.If not given, all classes are supposed to have weight one.The ""balanced"" mode uses the values of y to automatically adjustweights inversely proportional to class frequencies in the input dataas ``n_samples / (n_classes * np.bincount(y))``.Note that these weights will be multiplied with sample_weight (passedthrough the fit method) if sample_weight is specified... versionadded:: 0.17 *class_weight='balanced'*",None
,"random_state random_state: int, RandomState instance, default=NoneUsed when ``solver`` == 'sag', 'saga' or 'liblinear' to shuffle thedata. See :term:`Glossary ` for details.",None
,"solver solver: {'lbfgs', 'liblinear', 'newton-cg', 'newton-cholesky', 'sag', 'saga'}, default='lbfgs'Algorithm to use in the optimization problem. Default is 'lbfgs'.To choose a solver, you might want to consider the following aspects:- 'lbfgs' is a good default solver because it works reasonably well for a wide class of problems.- For :term:`multiclass` 

In [29]:
preds = model.predict(X_val)

from sklearn.metrics import accuracy_score
accuracy_score(y_val, preds)

0.988622754491018

### Hypertuning the naive bayes model

In [31]:
from sklearn.model_selection import GridSearchCV
from sklearn.naive_bayes import MultinomialNB
from sklearn.metrics import accuracy_score

param_grid = {
    'alpha': [0.01, 0.1, 0.5, 1.0],
    'fit_prior': [True, False]
}

nb_model = MultinomialNB()
grid_search = GridSearchCV(nb_model, param_grid, cv=5, scoring='f1', n_jobs=-1, verbose=1)
grid_search.fit(X_train, y_train)

print(grid_search.best_params_)
print(grid_search.best_score_)

model = grid_search.best_estimator_
preds = model.predict(X_val)
accuracy_score(y_val, preds)

Fitting 5 folds for each of 8 candidates, totalling 40 fits
{'alpha': 0.01, 'fit_prior': False}
0.9564005920420542


0.9784431137724551

In [33]:
test_df

,id,subject,email,text,clean_text
0,0,Subject: CERT Advisory CA-2002-21 Vulnerabilit...,\n \n -----BEGIN PGP SIGNED MESSAGE-----\n \n ...,Subject: CERT Advisory CA-2002-21 Vulnerabilit...,subject cert advisory ca vulnerability in php ...
1,1,Subject: ADV: Affordable Life Insurance ddbfk\n,Low-Cost Term-Life Insurance!\n SAVE up to 70%...,Subject: ADV: Affordable Life Insurance ddbfk\...,subject adv affordable life insurance ddbfk lo...
2,2,Subject: CAREER OPPORTUNITY. WORK FROM HOME\n,------=_NextPart_000_00A0_03E30A1A.B1804B54\n ...,Subject: CAREER OPPORTUNITY. WORK FROM HOME\n...,subject career opportunity work from home next...
3,3,Subject: Marriage makes both sexes happy\n,"URL: http://www.newsisfree.com/click/-3,848315...",Subject: Marriage makes both sexes happy\nURL:...,subject marriage makes both sexes happy url ht...
4,4,Subject: Re: [SAtalk] SA very slow (hangs?) on...,On Thursday 29 August 2002 16:39 CET Mike Burg...,Subject: Re: [SAtalk] SA very slow (hangs?) on...,subject re satalk sa very slow hangs on this m...
...,...,...,...,...,...
995,995,Subject: Re: Alsa/Redhat 8 compatability\n,"Once upon a time, Brian wrote :\n \n > \n > ...",Subject: Re: Alsa/Redhat 8 compatability\nOnce...,subject re alsaredhat compatability once upon ...
996,996,Subject: Re: Goodbye Global Warming\n,\n --]> A Green once said that if the Spotted ...,Subject: Re: Goodbye Global Warming\n\n --]> A...,subject re goodbye global warming a green once...
997,997,Subject: Re: Entrepreneurs\n,"On Fri, 23 Aug 2002, Robert Harley wrote:\n \n...","Subject: Re: Entrepreneurs\nOn Fri, 23 Aug 200...",subject re entrepreneurs on fri aug robert har...
998,998,Subject: Re: [ILUG] slashdot EW Dijkstra humor\n,JPL suggested:\n > Recursion is only truely u...,Subject: Re: [ILUG] slashdot EW Dijkstra humor...,subject re ilug slashdot ew dijkstra humor jpl...


In [34]:
X_test = vectorizer.transform(test_df['clean_text'])
results = model.predict(X_test)

In [35]:
submission_df = pd.DataFrame(zip(test_df["id"], results), columns=["id", "Class"])
submission_df["Class"] = submission_df["Class"].astype(int)
submission_df.to_csv("submission.csv", index=False)

In [37]:
submission_df.info()

<class 'pandas.DataFrame'>
RangeIndex: 1000 entries, 0 to 999
Data columns (total 2 columns):
 #   Column  Non-Null Count  Dtype
---  ------  --------------  -----
 0   id      1000 non-null   int64
 1   Class   1000 non-null   int64
dtypes: int64(2)
memory usage: 15.8 KB


In [38]:
import joblib

joblib.dump(model, 'spam_model.pkl')
joblib.dump(vectorizer, 'vectorizer.pkl')

['vectorizer.pkl']